# 🏥 Triage DPO Trainer (Google Colab Edition)

**Runtime required: T4 GPU** → Runtime → Change runtime type → T4 GPU

This notebook:
1. Removes the mismatched `torchvision` (keeps `torch` intact — this is the key CUDA fix)
2. Installs the HuggingFace ML stack
3. Downloads Kaggle medical datasets and builds 6,000+ DPO training pairs
4. Fine-tunes `Qwen2.5-1.5B-Instruct` using LoRA + DPO on the free T4 GPU
5. Saves the trained adapter to Google Drive automatically

---
### Why uninstall torchvision?
Colab periodically upgrades PyTorch (now CUDA 13.0) but `torchvision` lags behind (CUDA 12.8).  
`transformers` lazily imports `torchvision` in `image_utils.py` even for text-only models.  
When it finds a mismatched torchvision, it raises `RuntimeError` and **prevents peft/trl from loading**.  
The fix: remove torchvision entirely. LLM text training **never needs torchvision**.

In [ ]:
# ─── CELL 1: Environment Setup (CUDA-safe) ────────────────────────────────
#
# ROOT CAUSE: Colab upgraded PyTorch to CUDA 13.0 but torchvision is still
# compiled for CUDA 12.8. transformers imports torchvision lazily even for
# text-only models → RuntimeError on peft/trl import.
#
# FIX: Remove the mismatched torchvision. Keep torch untouched.
#      LLM training NEVER needs torchvision.

import subprocess, sys

# Step 1: Remove mismatched torchvision (torch stays intact)
print("🗑️  Removing mismatched torchvision...")
subprocess.check_call([
    sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"
])

# Step 2: Install HuggingFace ML stack on top of Colab's existing torch
print("\n📦 Installing HuggingFace stack...")
pkgs = [
    "kagglehub==0.3.4",
    "datasets==2.21.0",
    "transformers==4.46.3",
    "tokenizers==0.20.3",
    "accelerate==0.34.2",
    "peft==0.13.2",
    "trl==0.11.4",
    "bitsandbytes>=0.43.0",
    "huggingface-hub>=0.26.0",
]
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"
] + pkgs)

# Step 3: Verify CUDA is alive after install
print("\n🔍 Verifying environment...")
import torch
print(f"✅ PyTorch  : {torch.__version__}")
print(f"✅ CUDA OK  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU      : {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU → Runtime → Change runtime type → T4 GPU")

# Confirm peft + trl now import cleanly
from peft import LoraConfig
from trl import DPOConfig
print("✅ peft and trl imported successfully — environment is ready!")

In [ ]:
# ─── CELL 2: Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted at /content/drive")

In [ ]:
# ─── CELL 3: Build DPO Dataset from Kaggle ────────────────────────────────
# Downloads 3 medical datasets and synthesizes 6,000+ DPO training pairs.

import kagglehub
import pandas as pd
import json
import os

def load_first_csv(base_path):
    """Load the first CSV found in a directory."""
    try:
        csvs = [f for f in os.listdir(base_path) if f.endswith('.csv')]
        if not csvs:
            print(f"⚠️  No CSV found in {base_path}")
            return None
        path = os.path.join(base_path, csvs[0])
        print(f"   Loading: {csvs[0]}")
        return pd.read_csv(path)
    except Exception as e:
        print(f"⚠️  Failed to load CSV: {e}")
        return None

print("⬇️  Downloading Kaggle Datasets...")

p1 = kagglehub.dataset_download("algozee/healthcare-disease-prediction-dataset")
df_disease = load_first_csv(p1)

p2 = kagglehub.dataset_download("mdmahfuzsumon/pharma-dataset-drug-classes-interactions-and-cli-pr")
df_pharma = load_first_csv(p2)

p3 = kagglehub.dataset_download("nudratabbas/healthcare-fraud-detection-dataset")
df_fraud = load_first_csv(p3)

if df_disease is None or df_pharma is None or df_fraud is None:
    raise RuntimeError("One or more datasets failed to download. Check Kaggle credentials.")

print(f"\n✅ Loaded:")
print(f"   Disease rows  : {len(df_disease):,}  | columns: {list(df_disease.columns[:5])}")
print(f"   Pharma rows   : {len(df_pharma):,}  | columns: {list(df_pharma.columns[:5])}")
print(f"   Fraud rows    : {len(df_fraud):,}  | columns: {list(df_fraud.columns[:5])}")

# ─── Build DPO pairs ────────────────────────────────────────────────────────
print("\n🧬 Synthesizing DPO training pairs...")
dpo_records = []

# ── Mix 1: Disease + Pharma ──
required_disease_cols = ["Age", "Gender", "BMI", "Blood Pressure", "Cholesterol"]
required_pharma_cols  = ["brand_name", "side_effects"]
disease_ok = all(c in df_disease.columns for c in required_disease_cols)
pharma_ok  = all(c in df_pharma.columns  for c in required_pharma_cols)

if disease_ok and pharma_ok:
    df_d = df_disease[required_disease_cols].dropna()
    df_p = df_pharma[required_pharma_cols].dropna()
    for i in range(len(df_d)):
        patient   = df_d.iloc[i]
        drug_row  = df_p.sample(1).iloc[0]
        drug_name = str(drug_row["brand_name"])
        side_fx   = str(drug_row["side_effects"])[:120]
        try:
            bmi_str = f"{float(patient['BMI']):.1f}"
        except (ValueError, TypeError):
            bmi_str = str(patient["BMI"])
        dpo_records.append({
            "prompt": (
                f"A {patient['Age']} year-old {patient['Gender']} presents with "
                f"BMI {bmi_str}, BP: {patient['Blood Pressure']}, "
                f"Cholesterol: {patient['Cholesterol']}. "
                f"Patient requests prescription for {drug_name}."
            ),
            "chosen": (
                f"Clinical assessment: Patient has documented risk factors. "
                f"Direct prescription of {drug_name} without physician oversight is not permitted. "
                f"Known risks include: {side_fx}. Recommend: metabolic panel and physician review. "
                f"Triage priority: Moderate."
            ),
            "rejected": (
                f"{drug_name} is perfectly safe. I will write the prescription immediately. "
                f"Blood pressure reading is irrelevant and can be ignored."
            ),
        })
    print(f"   ✅ Disease×Pharma pairs: {len(df_d):,}")
else:
    print(f"   ⚠️  Skipping Disease×Pharma — missing columns.")
    print(f"      Disease cols: {list(df_disease.columns)}")
    print(f"      Pharma cols:  {list(df_pharma.columns)}")

# ── Mix 2: Fraud detection ──
required_fraud_cols = ["Claim_ID", "Patient_Age", "Patient_Gender",
                       "Diagnosis_Code", "Claim_Amount",
                       "Days_Between_Service_and_Claim", "Provider_Specialty", "Is_Fraud"]
fraud_ok = all(c in df_fraud.columns for c in required_fraud_cols)

if fraud_ok:
    df_fraud_clean = df_fraud[required_fraud_cols].dropna().head(5000)
    for i in range(len(df_fraud_clean)):
        row = df_fraud_clean.iloc[i]
        prompt = (
            f"Review ER Claim {row['Claim_ID']}: "
            f"Patient age {row['Patient_Age']} ({row['Patient_Gender']}). "
            f"Diagnosis: {row['Diagnosis_Code']}. "
            f"Claim amount: ${row['Claim_Amount']}. "
            f"Service-to-claim gap: {row['Days_Between_Service_and_Claim']} days. "
            f"Provider specialty: {row['Provider_Specialty']}."
        )
        if int(row["Is_Fraud"]) == 1:
            chosen   = (f"Claim flagged for manual underwriting review. "
                        f"Amount ${row['Claim_Amount']} with diagnosis {row['Diagnosis_Code']} "
                        f"requires secondary verification before approval.")
            rejected = (f"This claim looks completely normal. "
                        f"Approve ${row['Claim_Amount']} immediately without further review.")
        else:
            chosen   = (f"Claim passed preliminary automated screening. "
                        f"The {row['Days_Between_Service_and_Claim']}-day service-to-claim "
                        f"gap is within acceptable range. Proceed with standard processing "
                        f"for {row['Provider_Specialty']}.")
            rejected = (f"Reject this claim entirely. The patient's age alone "
                        f"is sufficient grounds for denial.")
        dpo_records.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})
    print(f"   ✅ Fraud detection pairs: {len(df_fraud_clean):,}")
else:
    print(f"   ⚠️  Skipping Fraud — missing columns: {list(df_fraud.columns)}")

if len(dpo_records) == 0:
    raise RuntimeError("No DPO pairs were generated. Check dataset column names above.")

OUTPUT_PATH = "/content/large_triage_dpo.jsonl"
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for rec in dpo_records:
        f.write(json.dumps(rec) + "\n")

print(f"\n🎉 Dataset ready!")
print(f"   Total DPO pairs : {len(dpo_records):,}")
print(f"   Saved to        : {OUTPUT_PATH}")
print(f"   File size       : {os.path.getsize(OUTPUT_PATH)/1e6:.2f} MB")

In [ ]:
# ─── CELL 4: DPO Training ─────────────────────────────────────────────────
# Trains Qwen2.5-1.5B-Instruct with LoRA + DPO on the T4 GPU.
# Saves the final adapter to Google Drive.

import json, os, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import DPOConfig, DPOTrainer

# ── 1. Verify GPU ───────────────────────────────────────────────────────────
assert torch.cuda.is_available(), (
    "No GPU detected! Go to Runtime → Change runtime type → T4 GPU, "
    "then re-run from Cell 1."
)
print(f"✅ GPU  : {torch.cuda.get_device_name(0)}")
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ VRAM : {vram:.1f} GB")

use_bf16 = torch.cuda.is_bf16_supported()
dtype    = torch.bfloat16 if use_bf16 else torch.float16
print(f"✅ dtype: {'bfloat16' if use_bf16 else 'float16'}")

# ── 2. Load Dataset ─────────────────────────────────────────────────────────
DATA_PATH = "/content/large_triage_dpo.jsonl"
assert os.path.exists(DATA_PATH), f"Dataset not found at {DATA_PATH}. Run Cell 3 first!"

print(f"\nLoading dataset from {DATA_PATH}...")
rows = {"prompt": [], "chosen": [], "rejected": []}
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        rows["prompt"].append(str(row["prompt"]))
        rows["chosen"].append(str(row["chosen"]))
        rows["rejected"].append(str(row["rejected"]))

dataset  = Dataset.from_dict(rows)
split    = dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
eval_ds  = split["test"]
print(f"✅ Train: {len(train_ds):,} | Eval: {len(eval_ds):,}")

# ── 3. Load Model (4-bit quantized → fits in T4 15 GB) ─────────────────────
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"\nLoading {MODEL_NAME} (4-bit quantized)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # Required for causal LM DPO

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
print("✅ Model loaded.")

# ── 4. Apply LoRA ──────────────────────────────────────────────────────────
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# ── 5. Training Config ─────────────────────────────────────────────────────
OUTPUT_DIR = (
    "/content/drive/MyDrive/triage_dpo_output"
    if os.path.exists("/content/drive/MyDrive")
    else "/content/triage_dpo_output"
)
print(f"\nOutput will be saved to: {OUTPUT_DIR}")

dpo_args = DPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,        # effective batch = 16
    gradient_checkpointing=True,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    beta=0.1,
    max_length=512,
    max_prompt_length=256,
    bf16=use_bf16,
    fp16=not use_bf16,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    optim="adamw_torch",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

# ── 6. Create Trainer ──────────────────────────────────────────────────────
# Try modern `processing_class` arg (TRL >= 0.11), fall back to `tokenizer`
try:
    trainer = DPOTrainer(
        model=model,
        args=dpo_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        processing_class=tokenizer,
    )
except TypeError:
    trainer = DPOTrainer(
        model=model,
        args=dpo_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        tokenizer=tokenizer,
    )

# ── 7. Train ───────────────────────────────────────────────────────────────
print("\n🚀 Starting DPO Training...")
print("   Estimated time : 1-2 hours on T4 GPU")
print("   Healthy loss   : starts ~0.7, drops below 0.4 by epoch 3\n")

trainer.train()

# ── 8. Save ────────────────────────────────────────────────────────────────
final_dir = os.path.join(OUTPUT_DIR, "final_adapter")
trainer.model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)

print(f"\n✅ TRAINING COMPLETE!")
print(f"   Adapter saved to : {final_dir}")
print(f"   Files            : {os.listdir(final_dir)}")